In [24]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CS340Project1 import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

USER = "aacuser"
PASS = "Jared1234"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Center(html.B(html.H1('Salvare Shelters'))),
    html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={
            'height': '300px', 
            'width': 'auto', 
            'display': 'block',
            'margin-left': 'auto',
            'margin-right': 'auto',
        }
    ),
    html.Hr(),
    
    # Interactive filtering options using Radio Buttons for Animal type and Rescue type
    
    # Animal Type Button Filters
    html.Div([
        html.H4("Filter by Animal Type:"),
        dcc.RadioItems(
            id='animal-type-filter',
            options=[
                {'label': 'Dogs', 'value': 'Dog'},
                {'label': 'Cats', 'value': 'Cat'},
                {'label': 'Other Animals', 'value': 'Other'},
                {'label': 'Reset Animal Filter', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        ),
        
        # Rescue Type Button Filters
        html.H4("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='rescue-type-filter',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster/Tracking Rescue', 'value': 'Disaster'},
                {'label': 'Reset Rescue Filter', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        ),
    ], style={'margin': '20px 0'}),

    html.Hr(),

    # Interactive Data Table
    html.H1("JAM's Dashboard"),
        dash_table.DataTable(
            id='datatable-id',
            columns=[
                {"name": i, 
                 "id": i, "deletable": False, "selectable": True} for i in df.columns
            ],
            data=df.to_dict('records'),
            
        # Features for interactive data table to make it user-friendly for your client
            page_size=10,
            filter_action="native",
            sort_action="native",
            row_selectable="single",
            selected_rows=[0],
            style_table={'overflowX': 'auto'},
            style_cell={
                'textAlign': 'left', 
                'minWidth': '100px', 
                'width': '150px', 
                'maxWidth': '200px', 
                'whiteSpace': 'normal'
            },
            style_header={
                'backgroundColor': 'rgb(230, 230, 230)', 
                'fontWeight': 'bold'
            },

        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

    
@app.callback(
    [Output('datatable-id', 'data'),
    Output('datatable-id', 'columns')],
    [Input('animal-type-filter', 'value'),
    Input('rescue-type-filter', 'value')])

def update_dashboard(animal_filter, rescue_filter):
    query = {} # base query
    
    # Animal Type Filter
    if animal_filter == 'Dog':
        query['animal_type'] = 'Dog'
    elif animal_filter == 'Cat':
        query['animal_type'] = 'Cat'
    elif animal_filter == 'Other':
        query['animal_type'] = {'$nin': ['Dog', 'Cat']}
        
    # Rescue Type Filter
    if rescue_filter == 'Water':
        query.update({
            'breed': {'$in': [
                'Labrador Retriever Mix', 
                'Chesapeake Bay Retriever', 
                'Newfoundland'
            ]},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        })
    elif rescue_filter == 'Mountain':
        query.update({
            'breed': {'$in': [
                'German Shepherd',
                'Alaskan Malamute',
                'Old English Sheepdog',
                'Siberian Husky',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        })
    elif rescue_filter == 'Disaster':
        query.update({
            'breed': {'$in': [
                'Doberman Pinscher',
                'German Shepherd',
                'Golden Retriever',
                'Bloodhound',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        })
    
    data = db.read(query)
    df = pd.DataFrame.from_records(data)
    df.drop(columns=['_id'], inplace=True)
    
    columns = [{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    return df.to_dict('records'), columns

# Pie Chart
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])

def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return html.Div("No data to display")
    
    dff = pd.DataFrame.from_dict(viewData)
    fig = px.pie(dff, names='breed', title='Breed Distribution')
    
    # Pie Chart Formatting 
    fig.update_layout(
        height=600,  
        width=800,   
        title_font_size=24,
        legend=dict(
            title="Breeds",
            font_size=14,
            orientation="v",  
            yanchor="middle",  
            y=0.5,           
            xanchor="left",   
            x=1.05,          
            itemwidth=30,     
            traceorder="normal"
        ),
        # setting a margin to prevent clipping between the legend and chart
        margin=dict(l=50, r=200, b=50, t=80),  
        showlegend=True
    )
    
    fig.update_traces(
        textposition='inside',
        textinfo='percent+label',
        textfont_size=12,
        marker=dict(line=dict(color='#000000', width=1)),
    )
    
    return [dcc.Graph(figure=fig)]

# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


app.run_server(mode='inline', debug=True)


---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

